<a href="https://colab.research.google.com/github/sarans2223/CNN/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub gradio

In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)
print(os.listdir(path))

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Path to dataset files: /kaggle/input/intel-image-classification
['seg_train', 'seg_pred', 'seg_test']


In [3]:
import os

for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    if level >= 2:
        continue

intel-image-classification/
  seg_train/
    seg_train/
      mountain/
      street/
      buildings/
      sea/
      forest/
      glacier/
  seg_pred/
    seg_pred/
  seg_test/
    seg_test/
      mountain/
      street/
      buildings/
      sea/
      forest/
      glacier/


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

print("TensorFlow version:", tf.__version__)

In [ ]:
train_dir = os.path.join(path, "seg_train", "seg_train")
test_dir = os.path.join(path, "seg_test", "seg_test")

print("Train directory:", train_dir)
print("Test directory:", test_dir)

print("Classes:")
print(os.listdir(train_dir))

In [ ]:
classes = sorted(os.listdir(train_dir))

train_counts = {}
test_counts = {}

for cls in classes:
    train_counts[cls] = len(os.listdir(os.path.join(train_dir, cls)))
    test_counts[cls] = len(os.listdir(os.path.join(test_dir, cls)))

print("Training images:")
print(train_counts)

print("\nTesting images:")
print(test_counts)

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(train_counts.keys(), train_counts.values())

plt.title("Training Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.figure(figsize=(15,10))

for i, cls in enumerate(classes):
    class_path = os.path.join(train_dir, cls)
    image_name = random.choice(os.listdir(class_path))
    image_path = os.path.join(class_path, image_name)

    image = Image.open(image_path)

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(cls)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
widths = []
heights = []

for cls in classes:
    class_path = os.path.join(train_dir, cls)

    for image_name in os.listdir(class_path)[:200]:
        image_path = os.path.join(class_path, image_name)

        try:
            img = Image.open(image_path)
            widths.append(img.width)
            heights.append(img.height)
        except:
            pass

print("Average width:", np.mean(widths))
print("Average height:", np.mean(heights))
print("Minimum width:", np.min(widths))
print("Maximum width:", np.max(widths))
print("Minimum height:", np.min(heights))
print("Maximum height:", np.max(heights))

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
class_names = train_ds.class_names

print(class_names)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
model = Sequential([

    # Data augmentation
    data_augmentation,

    # Normalize pixels
    layers.Rescaling(1./255),

    # Convolution Block 1
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Convolution Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Convolution Block 3
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Convolution Block 4
    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    # Flatten
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(256, activation='relu'),

    # Dropout
    layers.Dropout(0.5),

    # Output layer
    layers.Dense(6, activation='softmax')
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
import os
import kagglehub

# Fallback dataset path and definitions if not in memory
if 'path' not in globals():
    path = kagglehub.dataset_download("puneet6060/intel-image-classification")

if 'train_ds' not in globals() or 'val_ds' not in globals():
    train_dir = os.path.join(path, "seg_train", "seg_train")
    test_dir = os.path.join(path, "seg_test", "seg_test")
    IMG_SIZE = (150, 150)
    BATCH_SIZE = 32
    SEED = 42

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Ensure data augmentation is defined
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

# Ensure model is defined and compiled before fitting
model = Sequential([
    data_augmentation,
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("CNN Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title("CNN Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = model.predict(images, verbose=0)

    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',

    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")

plt.show()

In [ ]:
def predict_image(image_path):

    img = tf.keras.utils.load_img(
        image_path,
        target_size=IMG_SIZE
    )

    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array, verbose=0)

    predicted_index = np.argmax(predictions[0])
    confidence = np.max(predictions[0]) * 100

    predicted_class = class_names[predicted_index]

    return predicted_class, confidence

In [ ]:
sample_class = random.choice(class_names)

sample_folder = os.path.join(train_dir, sample_class)

sample_image = random.choice(os.listdir(sample_folder))

sample_path = os.path.join(sample_folder, sample_image)

prediction, confidence = predict_image(sample_path)

print("Actual Class:", sample_class)
print("Predicted Class:", prediction)
print("Confidence:", round(confidence, 2), "%")

plt.imshow(Image.open(sample_path))
plt.axis("off")
plt.show()

In [ ]:
model.save("/content/intel_scene_classifier.keras")

In [ ]:
os.path.exists("/content/intel_scene_classifier.keras")

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

In [ ]:
def classify_scene(image):

    image = image.resize(IMG_SIZE)

    img_array = np.array(image)

    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]

    results = {
        class_names[i]: float(predictions[i])
        for i in range(len(class_names))
    }

    return results

In [ ]:
demo = gr.Interface(
    fn=classify_scene,
    inputs=gr.Image(type="pil", label="Upload Scene Image"),
    outputs=gr.Label(num_top_classes=6, label="Prediction"),
    title="🌍 Intel Scene Classification using CNN",
    description=(
        "Upload an image and the CNN model will classify it "
        "as Buildings, Forest, Glacier, Mountain, Sea, or Street."
    )
)

In [ ]:
demo.launch(share=True)